# Loan Default Risk with Business Cost Optimization

## Objective
Predict loan default and optimize threshold to minimize business cost.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.impute import SimpleImputer


In [ ]:
df = pd.read_csv("application_train.csv")
df.head()

In [ ]:
missing = df.isnull().mean()
df = df.loc[:, missing < 0.5]

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
df = pd.get_dummies(df, drop_first=True)

X = df.drop("TARGET", axis=1)
y = df["TARGET"]

In [ ]:
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

X = pd.DataFrame(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
cost_fp = 1
cost_fn = 5

thresholds = np.arange(0.1,0.9,0.05)
costs = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_cost = (fp*cost_fp) + (fn*cost_fn)
    costs.append(total_cost)

In [ ]:
plt.plot(thresholds, costs)
plt.xlabel("Threshold")
plt.ylabel("Cost")
plt.show()

In [ ]:
best_threshold = thresholds[np.argmin(costs)]
print("Best Threshold:", best_threshold)

In [ ]:
y_pred_final = (y_prob >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_final))